# Clean MNIST geometry: Does a pretrained EBM actually help?

**One decisive experiment — no Gaussian pixel noise, no model retraining, no Hessian eigengap threshold.**

This tests a *specific, falsifiable hypothesis*: Does weighting multiscale local PCA by the squared Hessian of a frozen pretrained MNIST EBM improve geometry of unseen clean handwritten `1`s?

**Three safeguards:**
1. First verify the method against a known 2-dimensional sphere with an *analytic oracle* energy. This is not a learned-EBM test.
2. Train local covariance subspaces on one group of clean digit-1 images; evaluate the subspaces on *different, withheld, unmodified* nearby digit-1 images. The UvA EBM is pretrained on all MNIST digits, frozen.
3. Compare `gamma=0` (pure local PCA) with `gamma=1,4` (energy-guided). Use a **predeclared go/no-go test** rather than tuning thresholds against the paper's digit-1 estimate of 66.

**What this cannot guarantee:** The experiment does **not** prove a unique intrinsic dimension for real-world MNIST. It tests whether the frozen EBM makes *candidate tangent directions* more predictive of independent same-class data. The multiscale scaling modes and kNN-MLE estimates are exploratory, not ground truth.

Official pretrained checkpoint and model architecture: [UvA Tutorial 8](https://uvadlc-notebooks.readthedocs.io/en/latest/tutorial_notebooks/tutorial8/Deep_Energy_Models.html).


## Mathematical method

For clean neighbors around image $x$, form covariance $C_x(r)$. With frozen EBM energy $E_\theta(x)$ and Hessian $H_x=\nabla^2_x E_\theta(x)$, define

$$M_x=I+\gamma(H_x/s_x)^2,\qquad s_x^2=\operatorname{tr}(H_x^2)/784.$$

Solve the generalized eigenproblem

$$C_x(r)v_j=\mu_j(r)M_x v_j.$$

When $\gamma=0$, this is ordinary PCA; otherwise the penalty discourages high energy-curvature directions. Squaring the Hessian keeps $M_x$ positive definite even if the energy Hessian has negative eigenvalues. The method uses the **full Hessian only on MNIST** to avoid another confounding numerical SLQ error; it is cached per image.

The independent validation score is

$$A_d=\mathbb E_{y\,\text{held out}}\frac{\|P_d(y-x)\|_2^2}{\|y-x\|_2^2}.$$

Here $P_d$ is the Euclidean projector onto a $d$-dimensional candidate subspace (the **rank $d$ is selected for evaluation**, not claimed as MNIST's intrinsic dimension). The random-subspace reference is $d/784$.

We also examine the scaling law $\mu_j(r)\propto r^{\alpha_j}$, but only if the neighborhood radii span enough scales. Tangent-like modes often scale as $r^2$ under restrictive smooth manifold assumptions. If the range is insufficient, the script reports that the dimension is **not identifiable**, rather than inventing a rank.

## Step 1 — Set up CPU execution and write the complete standalone script

In Colab, use **Runtime → Run all**. No GPU required; CUDA is not used because we want a consistent small MNIST baseline.

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
import numpy, torch, sklearn, pandas, torchvision, matplotlib
print("PyTorch:", torch.__version__, "Torchvision:", torchvision.__version__)
print("CPU-ready. No extra pip installs necessary in standard Colab.")

In [ ]:
%%writefile /content/multiscale_ebm_decisive.py
#!/usr/bin/env python3
"""Clean-data multiscale geometry: a falsifiable MNIST experiment.

No added pixel noise, no EBM training, no tuning to published intrinsic dimensions.
Metric M=I + gamma (H/s)^2, generalized covariance C v = mu M v.
For gamma=0 this is local PCA. The generalized eigenvalue slopes vs radius
are diagnostic only; don't treat slope-count as established manifold dimension.
"""
import argparse
import json
import math
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.neighbors import NearestNeighbors
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

D_MNIST = 784
CKPT_URL = 'https://raw.githubusercontent.com/phlippe/saved_models/main/tutorial8/MNIST.ckpt'
PAPER = {0:113,1:66,2:131,3:120,4:107,5:129,6:126,7:100,8:148,9:152}

class Swish(nn.Module):
    def forward(self,x):return x*torch.sigmoid(x)

class CNNModel(nn.Module):
    # Exact tutorial8 architecture; outputs f(x)=-E(x); H^2 sign invariant.
    def __init__(self,hidden_features=32):
        super().__init__()
        a=hidden_features//2;b=hidden_features;c=hidden_features*2
        self.cnn_layers=nn.Sequential(
            nn.Conv2d(1,a,kernel_size=5,stride=2,padding=4),Swish(),
            nn.Conv2d(a,b,kernel_size=3,stride=2,padding=1),Swish(),
            nn.Conv2d(b,c,kernel_size=3,stride=2,padding=1),Swish(),
            nn.Conv2d(c,c,kernel_size=3,stride=2,padding=1),Swish(),
            nn.Flatten(),nn.Linear(c*4,c),Swish(),nn.Linear(c,1))
    def forward(self,x):return self.cnn_layers(x).squeeze(-1)

def get_ebm(path,device='cpu'):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if not path.exists():
        print('Downloading frozen UvA MNIST model...',flush=True)
        urllib.request.urlretrieve(CKPT_URL,path)
    try: ckpt=torch.load(path,map_location='cpu',weights_only=True)
    except Exception as e:
        raise RuntimeError('Could not safely read checkpoint; check the official download') from e
    sd=ckpt['state_dict'] if isinstance(ckpt,dict) and 'state_dict' in ckpt else ckpt
    sd={k[4:] if k.startswith('cnn.') else k:v for k,v in sd.items()
        if k.startswith('cnn.') or k.startswith('cnn_layers.')}
    first=sd['cnn_layers.0.weight'].shape[0]
    net=CNNModel(hidden_features=first*2).to(device=device,dtype=torch.float64)
    net.load_state_dict(sd,strict=True)
    net.eval()
    for p in net.parameters():p.requires_grad_(False)
    return net

def full_hessian(net, image):
    # image in [0,1] pixels; UvA network uses [-1,1].
    # Hessian wrt [-1,1] pixels. Relative normalization makes constant scaling irrelevant.
    x=torch.as_tensor(image.reshape(-1)*2-1,dtype=torch.float64)
    with torch.enable_grad():
        z=x.detach().clone().requires_grad_(True)
        out=net(z.reshape(1,1,28,28)).sum()
        grad=torch.autograd.grad(out,z,create_graph=True)[0]
        cols=[]
        # Each column is an exact HVP; no construction of a full fourth-order tensor.
        for j in range(D_MNIST):
            v=torch.zeros_like(z);v[j]=1.
            col=torch.autograd.grad(grad[j],z,retain_graph=j<D_MNIST-1)[0].detach()
            cols.append(col)
    H=torch.stack(cols,dim=1).numpy()
    H=.5*(H+H.T)
    vals,U=np.linalg.eigh(H)
    scale=np.sqrt(np.mean(vals**2))
    return vals,U,scale

def get_mnist_arrays(root):
    from torchvision.datasets import MNIST
    train=MNIST(root=root,train=True,download=True)
    test=MNIST(root=root,train=False,download=True)
    trX=train.data.numpy().reshape(-1,D_MNIST).astype(np.float32)/255.
    teX=test.data.numpy().reshape(-1,D_MNIST).astype(np.float32)/255.
    return trX,train.targets.numpy(),teX,test.targets.numpy()

def get_neighborhood(reference_X, x, k_values, clean_holdout=False):
    """Select independent, similarly local, clean same-class reference samples.

    Interleaving by distance gives fit and test neighbors with comparable radii.
    We never fit the covariance with the held-out samples.
    """
    kmax = max(k_values)
    n_req = 2 * kmax if clean_holdout else kmax
    nbrs = NearestNeighbors(n_neighbors=n_req, algorithm='brute',
                            metric='euclidean', n_jobs=1)
    nbrs.fit(reference_X)
    dist, idx = nbrs.kneighbors(x.reshape(1, -1), return_distance=True)
    if clean_holdout:
        fit_idx, fit_dist = idx[0, ::2], dist[0, ::2]
        hold_idx = idx[0, 1::2]
        return (reference_X[fit_idx].astype('float64'),
                fit_dist.astype('float64'),
                reference_X[hold_idx].astype('float64'),
                dist[0].astype('float64'))
    return reference_X[idx[0]].astype('float64'), dist[0].astype('float64')

def mle_from_knn(dist,k):
    dist=np.asarray(dist)
    k=min(k,len(dist))
    if k<5 or dist[0] < 1e-10: return float('nan')
    rk=dist[k-1]
    ratio=np.log(np.maximum(rk,1e-15)/np.maximum(dist[:k-1],1e-15))
    return float(1./np.mean(ratio)) if np.mean(ratio)>1e-8 else float('nan')

def multiscale(neighbors,dist,evals,U,k_values,gammas,rank_max=96):
    """Sample-space generalized eigensystem with shared, frozen full Hessian.

    For gamma>=0: M=I+gamma*(H/s)^2. If W=(X-mu).T, nonzero
    eigenvalues of M^-1/2 C M^-1/2 equal eigs of W^T M^-1 W/k.
    We use H eigenvectors U to diagonalize the metric once per image.
    """
    D=neighbors.shape[1]
    assert all(2 < k <= len(dist) for k in k_values)
    if evals is None:
        U=np.eye(D)
        h2=np.zeros(D)
    else:
        sc=np.sqrt(np.mean(evals**2))
        h2=(evals/max(sc,1e-18))**2
    projected=U.T @ neighbors.T  # D x K; reuse for all k
    results={}; spec_rows=[]
    for gamma in gammas:
        Minv=1./(1.+gamma*h2)
        records=[]
        for k in k_values:
            Y=projected[:,:k].copy()
            Y-=Y.mean(axis=1,keepdims=True)
            whitened=Y*np.sqrt(Minv[:,None])
            G=whitened.T@whitened/k
            eig,A=np.linalg.eigh(G)
            order=np.argsort(eig)[::-1]
            eig=np.maximum(0.,eig[order]);A=A[:,order]
            valid=min(rank_max,k-1)
            e=eig[:valid]
            # v_j=M^-1 W a_j/sqrt(k*mu_j): generalized eigenvectors.
            a=A[:,:valid]
            V=U @ (Minv[:,None]*(Y@a))
            norm=np.linalg.norm(V,axis=0)
            V=V/np.maximum(norm,1e-16)
            records.append(dict(k=k,radius=float(dist[k-1]),eigen=e,vec=V))
        radii=np.asarray([r['radius'] for r in records]);lr=np.log(np.maximum(radii,1e-14))
        span=float(lr.max()-lr.min())
        upto=min(len(r['eigen']) for r in records)
        slopes=np.full(upto,np.nan);fits=np.full(upto,np.nan)
        for j in range(upto):
            vals=np.asarray([r['eigen'][j] for r in records])
            # Require eigenvalues safely above numerical zero for all radii.
            if np.any(vals<=1e-10*max(1.,records[-1]['eigen'][0])) or span<0.12:
                continue
            lv=np.log(vals)
            slope,intercept=np.polyfit(lr,lv,1)
            slopes[j]=slope
            fits[j]=float(np.sqrt(np.mean((lv-(slope*lr+intercept))**2)))
        # Only report a *heuristic* tangent-like slope count, not ground-truth LID.
        eligible=np.isfinite(slopes)&(np.abs(slopes-2.)<=0.7)&(fits<=0.22)
        # leading contiguous modes, preventing arbitrary distant ranks.
        prefix=0
        for good in eligible:
            if not good:break
            prefix+=1
        # fixed-scale participation ratio (always a descriptive metric)
        ee=records[-2]['eigen']
        pr=(ee.sum()**2/max(np.dot(ee,ee),1e-15))
        results[gamma]=dict(records=records,slopes=slopes,fit_rms=fits,
                             log_radius_span=span,scaling_prefix=int(prefix),
                             scaling_identifiable=bool(span>=.12),
                             participation_ratio=float(pr))
        for j in range(min(upto,rank_max)):
            spec_rows.append(dict(gamma=gamma,rank=j+1,scaling_slope=slopes[j],
                                  fit_rms=fits[j],log_radius_span=span))
    return results,pd.DataFrame(spec_rows)

def tangent_alignment(V,vectors,ranks):
    """Ambient Euclidean projection, even for generalized eigenvectors."""
    ans={}
    for rank in ranks:
        rank=min(rank,V.shape[1])
        basis=np.linalg.qr(V[:,:rank],mode='reduced')[0]
        vals=[]
        for v in vectors:
            v=np.asarray(v,dtype='float64');v/=max(np.linalg.norm(v),1e-16)
            vals.append(float(np.sum((basis.T@v)**2)))
        ans[rank]=float(np.mean(vals))
    return ans

def synthetic_example(seed=21,N=3500,D=32,k_values=(64,96,160,256),gamma_values=(0.,1.,4.),q=8):
    rng=np.random.default_rng(seed)
    embedding,_=np.linalg.qr(rng.standard_normal((D,D)))
    sphere3=embedding[:,:3]
    u=rng.standard_normal((N,3));u/=np.linalg.norm(u,axis=1,keepdims=True)
    X=u@sphere3.T
    queries=rng.standard_normal((q,3));queries/=np.linalg.norm(queries,axis=1,keepdims=True)
    rows=[]; diag=[]
    for qi,p in enumerate(queries):
        x=sphere3@p
        t1=rng.standard_normal(3);t1-=p*np.dot(t1,p);t1/=np.linalg.norm(t1)
        t2=np.cross(p,t1)
        T=sphere3@np.stack([t1,t2],axis=1)
        # Squared distance to sphere in R^D has H=I-P_T on manifold.
        H=np.eye(D)-T@T.T
        evals,U=np.linalg.eigh(H)
        neigh,dist=get_neighborhood(X.astype('float32'),x.astype('float32'),k_values)
        estimates,spec=multiscale(neigh,dist,evals,U,k_values,gamma_values,rank_max=12)
        for gamma,v in estimates.items():
            # Use same k and 2-D basis for fair comparison.
            V=v['records'][-2]['vec']
            Q=np.linalg.qr(V[:,:2])[0]
            P=Q@Q.T;Pt=T@T.T
            err=float(np.linalg.norm(P-Pt,'fro')/np.sqrt(2*2))
            rows.append(dict(query=qi,gamma=gamma,true_dimension=2,
                             tangent_projector_error=err,
                             slope_prefix=v['scaling_prefix'],
                             log_radius_span=v['log_radius_span'],
                             participation_ratio=v['participation_ratio']))
        diag.append(spec.assign(query=qi))
    return pd.DataFrame(rows),pd.concat(diag,ignore_index=True)

def decide(synthetic,mnist,goal=0.005):
    out=[]
    syn0=synthetic.loc[synthetic.gamma==0].set_index('query')
    for gamma in sorted(set(synthetic.gamma)-{0.}):
        s=synthetic.loc[synthetic.gamma==gamma].set_index('query')
        improvement=(syn0.tangent_projector_error-s.tangent_projector_error)
        out.append(dict(dataset='synthetic',gamma=gamma,
                        mean_improvement=float(improvement.mean()),
                        fraction_improved=float((improvement>1e-8).mean()),
                        passes=bool(improvement.mean()>0.005 and (improvement>0).mean()>=.75)))
    if not mnist.empty:
        baseline=mnist.loc[(mnist.gamma==0)&(mnist['rank']==32)].set_index('test_index')
        for gamma in sorted(set(mnist.gamma)-{0.}):
            s=mnist.loc[(mnist.gamma==gamma)&(mnist['rank']==32)].set_index('test_index')
            delta=s.alignment-baseline.alignment
            out.append(dict(dataset='MNIST_CLEAN_HELDOUT',gamma=gamma,
                            mean_improvement=float(delta.mean()),
                            fraction_improved=float((delta>0).mean()),
                            passes=bool(delta.mean()>=goal and (delta>0).mean()>=2/3)))
    return pd.DataFrame(out)

def plot_summary(synthetic,mnist,out):
    fig,axs=plt.subplots(1,2,figsize=(11,4))
    a=synthetic.groupby('gamma').tangent_projector_error.agg(['mean','std'])
    axs[0].bar([str(v) for v in a.index],a['mean'],yerr=a['std'].fillna(0),capsize=3)
    axs[0].set_title('Known 2-D sphere: projector error (lower better)')
    axs[0].set_xlabel('EBM guidance gamma');axs[0].set_ylabel('normalized Frobenius error')
    if len(mnist):
        b=mnist[mnist['rank']==32].groupby('gamma').alignment.agg(['mean','std'])
        axs[1].bar([str(v) for v in b.index],b['mean'],yerr=b['std'].fillna(0),capsize=3)
        axs[1].axhline(32/784,linestyle='--',label='Random 32-D subspace expectation')
        axs[1].legend(fontsize=7)
    axs[1].set_title('Clean MNIST: held-out neighbor alignment (higher better)')
    axs[1].set_xlabel('EBM guidance gamma');axs[1].set_ylabel('captured directional energy')
    for ax in axs:ax.grid(alpha=.2,axis='y')
    fig.tight_layout();fig.savefig(out/'decision_comparison.png',dpi=150);plt.close(fig)

def run(args):
    torch.set_num_threads(max(1,args.torch_threads))
    try:
        from threadpoolctl import threadpool_limits
        threadpool_limits(limits=max(1,args.torch_threads))
    except ImportError:
        pass
    out=Path(args.out);out.mkdir(parents=True,exist_ok=True)
    kv=tuple(int(v) for v in args.k_values.split(','))
    gammas=tuple(float(v) for v in args.gammas.split(','))
    if 0. not in gammas:raise ValueError('gammas must contain 0 (no-EBM baseline)')
    if len(kv)<4 or sorted(kv)!=list(kv):raise ValueError('Use >=4 increasing k values')
    start=time.perf_counter()
    print('STAGE 1: Known 2-D sphere, analytic squared-distance energy.',flush=True)
    synthetic,synspec=synthetic_example(N=args.synthetic_n,q=args.synthetic_queries,
                                        k_values=kv,gamma_values=gammas)
    synthetic.to_csv(out/'synthetic_results.csv',index=False)
    synspec.to_csv(out/'synthetic_scaling_modes.csv',index=False)
    print(synthetic.groupby('gamma')[['tangent_projector_error','slope_prefix','log_radius_span']].agg(['mean','std']).round(3),flush=True)
    rows=[];modes=[]
    if not args.synthetic_only:
        from torchvision.datasets import MNIST
        print('STAGE 2: Clean MNIST, unseen test digits, pretrained EBM.',flush=True)
        trainX,trainY,testX,testY=get_mnist_arrays(args.data_dir)
        ids=[int(t) for t in args.indices.split(',')] if args.indices.strip() else \
             list(np.flatnonzero(testY==args.digit)[::max(1,int(np.sum(testY==args.digit)/12))][:12])
        reference=trainX[trainY==args.digit]
        ebm=get_ebm(args.checkpoint)
        cache=out/'hessian_cache';cache.mkdir(exist_ok=True)
        for idx in ids:
            if int(testY[idx])!=args.digit:
                raise ValueError(f'Test index {idx} label {int(testY[idx])}, expected {args.digit}')
            x=testX[idx].astype('float64')
            t0=time.perf_counter()
            path=cache/f'full_hessian_{idx}.npz'
            if path.exists() and not args.recompute:
                dat=np.load(path);eig=dat['eig'];U=dat['U']
            else:
                eig,U,scale=full_hessian(ebm,x)
                np.savez_compressed(path,eig=eig,U=U)
            neighbors,dist,heldout,all_dist=get_neighborhood(reference,x.astype('float32'),kv,clean_holdout=True)
            res,curves=multiscale(neighbors,dist,eig,U,kv,gammas,rank_max=max(args.ranks))
            modes.append(curves.assign(test_index=idx,digit=args.digit))
            # Primary independent validation uses only *unmodified clean* train images
            # withheld from fitting the local covariance. No artificial transformations.
            heldout_directions=heldout[:min(args.heldout_n,len(heldout))]-x.reshape(1,-1)
            for gamma,model in res.items():
                # k scale: second largest, fixed before seeing results
                V=model['records'][-2]['vec']
                align=tangent_alignment(V,heldout_directions,args.ranks)
                for rank,score in align.items():
                    rows.append(dict(test_index=idx,digit=args.digit,gamma=gamma,rank=rank,
                                     alignment=score,random_reference=rank/784,
                                     scaling_prefix=model['scaling_prefix'],
                                     log_radius_span=model['log_radius_span'],
                                     participation_ratio=model['participation_ratio'],
                                     mle_k20=mle_from_knn(all_dist,20),
                                     mle_k40=mle_from_knn(all_dist,40),
                                     validation='independent clean same-class neighbors',
                                     paper_diffusion_reference=PAPER[args.digit]))
            print(f'  image {idx}: {time.perf_counter()-t0:.1f}s, '
                  f'log radius span {res[0.]["log_radius_span"]:.3f}, '
                  f'scaling prefix {res[0.]["scaling_prefix"]}',flush=True)
        pd.concat(modes,ignore_index=True).to_csv(out/'mnist_scaling_modes.csv',index=False)
    mnist=pd.DataFrame(rows)
    mnist.to_csv(out/'mnist_alignment.csv',index=False)
    decisions=decide(synthetic,mnist)
    decisions.to_csv(out/'predeclared_go_no_go.csv',index=False)
    plot_summary(synthetic,mnist,out)
    info={'description':'Exploratory test; no estimated MNIST manifold LID is claimed',
          'dataset':'clean MNIST held-out examples, same-class training neighbors',
          'gamma_values':gammas,'k_values':kv,'predeclared_alignment_ranks':args.ranks,
          'heldout_neighbors_per_anchor':args.heldout_n,
          'training_noise':False,'added_image_noise':False,
          'elapsed_seconds':round(time.perf_counter()-start,1),
          'interpretation':'No claimed MNIST manifold dimension; slope/participation and MLE are descriptive diagnostics',
          'gate':'Continue EBM research only if synthetic ground truth AND clean held-out MNIST alignment improve; else use the no-EBM baseline'}
    (out/'method_notes.json').write_text(json.dumps(info,indent=2))
    print('\nPREDECLARED GO/NO-GO ASSESSMENT:\n'+decisions.to_string(index=False),flush=True)
    synthetic_ok = set(decisions.loc[(decisions.dataset=='synthetic') & decisions.passes,'gamma'])
    mnist_ok = set(decisions.loc[(decisions.dataset=='MNIST_CLEAN_HELDOUT') & decisions.passes,'gamma'])
    if args.synthetic_only:
        verdict = 'SYNTHETIC CONTROL ONLY: No conclusion about the real pretrained MNIST EBM.'
    elif synthetic_ok & mnist_ok:
        verdict = ('GO: EBM-guided local geometry improves the predeclared synthetic '
                   'and independent clean MNIST tests. Continue geometric validation; '
                   'this is NOT evidence of a correct intrinsic dimension.')
    else:
        verdict = ('NO-GO for this frozen EBM as a manifold-tangent prior under '
                   'these predeclared tests. Keep the clean-data PCA baseline; '
                   'do not tune Hessian thresholds to manufacture an ID.')
    (out/'FINAL_DECISION.txt').write_text(verdict+'\n')
    print('\n'+verdict+'\n',flush=True)
    if not mnist.empty:
        print('\nMNIST rank-32 alignment on withheld clean neighbor images:\n'+
              mnist[mnist['rank']==32].pivot(index='test_index',columns='gamma',values='alignment').round(3).to_string(),flush=True)
    print('\nOutput',out.resolve(),flush=True)

if __name__=='__main__':
    p=argparse.ArgumentParser()
    p.add_argument('--synthetic-only',action='store_true')
    p.add_argument('--synthetic-n',type=int,default=3500)
    p.add_argument('--synthetic-queries',type=int,default=8)
    p.add_argument('--digit',type=int,default=1)
    p.add_argument('--indices',default='',help='Optional comma-separated test indices; empty selects 12 class-specific test images')
    p.add_argument('--k-values',default='48,96,160,256')
    p.add_argument('--gammas',default='0,1,4')
    p.add_argument('--ranks',default='16,32,64')
    p.add_argument('--heldout-n',type=int,default=128)
    p.add_argument('--data-dir',default='/content/mnist_data')
    p.add_argument('--checkpoint',default='/content/mnist_ebm_weights/MNIST.ckpt')
    p.add_argument('--out',default='/content/multiscale_ebm_results')
    p.add_argument('--torch-threads',type=int,default=2)
    p.add_argument('--recompute',action='store_true')
    args=p.parse_args()
    args.ranks=tuple(int(r) for r in args.ranks.split(','))
    run(args)


## Step 2 — Run the *complete* decisive experiment

The program downloads the frozen UvA model once and caches each full Hessian. It evaluates **12 predetermined clean test images of digit 1**, trains on nearest class-1 training neighbors, and tests with **withheld independent clean neighbors**. For each query it evaluates the same candidate dimensions 16, 32, 64 for all methods. The pass/fail rule is set **before** seeing results.

After running, do **not** change `gamma`, neighborhood sizes, or threshold criteria merely to make the result look favorable.

In [ ]:
!python /content/multiscale_ebm_decisive.py \
    --digit 1 \
    --k-values 48,96,160,256 \
    --gammas 0,1,4 \
    --ranks 16,32,64 \
    --heldout-n 128 \
    --synthetic-queries 8 \
    --synthetic-n 3500 \
    --torch-threads 2 \
    --out /content/ebm_clean_geometry_decision

## Step 3 — Read the decision, not another spectrum plot

- **GO** requires an EBM-guided $\gamma$ to improve the known synthetic projector test **and** the clean held-out MNIST alignment. The MNIST criterion is ≥ 0.005 mean absolute improvement at rank 32, with > two thirds of test images improved.
- **NO-GO** means the current frozen EBM Hessian is not a useful additional tangent prior *for this experiment*. Stop tuning it; use the no-EBM baseline or switch to a model whose geometry is independently supported.
- The synthetic *oracle* control is not evidence that the pretrained EBM learned the same geometry.

No comparison against a published MNIST intrinsic dimension is used to pick the result.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
out = Path('/content/ebm_clean_geometry_decision')
print((out/'FINAL_DECISION.txt').read_text())
print('\nPredeclared tests:')
display(pd.read_csv(out/'predeclared_go_no_go.csv'))
print('\nRank 32: clean held-out neighbor alignment; higher is better')
m = pd.read_csv(out/'mnist_alignment.csv')
display(m[m['rank']==32].pivot(index='test_index',columns='gamma',values='alignment').round(4))
print('\nSynthetic true tangent projector error; lower is better')
s = pd.read_csv(out/'synthetic_results.csv')
display(s.groupby('gamma',as_index=False).agg(mean_projector_error=('tangent_projector_error','mean'),mean_scaling_prefix=('slope_prefix','mean')).round(4))
display(Image(filename=str(out/'decision_comparison.png')))

## Step 4 — Inspect dimension diagnostics responsibly

`scaling_prefix` counts stable *leading scaling modes* only when the radii span enough scales. `participation_ratio` and kNN-MLE are descriptive estimates, **not** independently validated MNIST intrinsic dimensions. In particular, if the output says `scaling_identifiable=False` or slope prefix is zero, **do not reinterpret 0 as the true intrinsic dimension**. The paper's digit-wise diffusion results are included as external references, not fitting targets.

In [ ]:
print('Clean MNIST diagnostics (rank=32, no EBM penalty):')
subset = m[(m['gamma']==0)&(m['rank']==32)]
display(subset[['test_index','scaling_prefix','log_radius_span','participation_ratio','mle_k20','mle_k40','paper_diffusion_reference']].round(3))
import shutil
archive = shutil.make_archive('/content/ebm_clean_geometry_decision','zip',root_dir=out)
print('Results archive:',archive)
print('To download manually in Colab, open the Files sidebar and select this archive.')

## What happens after the decision?

If **GO**, run the same locked procedure on more test samples and a second independently trained EBM; then validate transformations and projector quality. If **NO-GO**, **stop this EBM/Hessian line** and continue with pure clean-data neighborhood methods or a score model with a demonstrated tangent/normal theorem. A negative experiment is a valid stopping point—not an invitation to relabel eigenvalue counts as dimensions.

**Reproducibility:** no training, no random pixel corruption, no hidden labels used in geometric fitting (class labels only select reference samples), no manual cutoff chasing. All CSVs and cached full-Hessian eigendecompositions are stored under `/content/ebm_clean_geometry_decision`.